In [ ]:
!pip install biopython requests

In [ ]:
!apt-get update


In [ ]:
!wget https://github.com/shenwei356/seqkit/releases/download/v2.6.1/seqkit_linux_amd64.tar.gz


In [5]:
!tar -xzf seqkit_linux_amd64.tar.gz

In [6]:
!mv seqkit /usr/local/bin/


In [7]:
!chmod +x /usr/local/bin/seqkit

In [ ]:
!seqkit version

In [ ]:
# Uploading necessary files to Colab environment

from google.colab import files

uploaded = files.upload()

In [22]:
#!/usr/bin/env python3
from Bio import SeqIO
from Bio.SeqUtils import molecular_weight
import hashlib
import statistics
import re
import requests
import sys
import json

class MyFastaParser:
    def __init__(self, fasta_file):
        self.fasta_file = fasta_file
        self._warning = set()

    def seqkit_stats(self):
        """Calculate statistics and detect file type"""
        seq_lens = []
        num_seqs = 0
        sum_gap = 0

        # Auto-detect file type by analyzing sequences
        protein_chars = set('ACDEFGHIKLMNPQRSTVWY')
        dna_chars = set('ATCG')

        dna_count = 0
        protein_count = 0

        for record in SeqIO.parse(self.fasta_file, "fasta"):
            seq = str(record.seq).upper().replace('-', '').replace('*', '')
            num_seqs += 1
            seq_lens.append(len(record.seq))
            sum_gap += str(record.seq).count("-")

            # Analyze all sequences for type detection
            if len(seq) > 0:
                seq_set = set(seq)

                # Check if sequence contains only DNA bases
                if seq_set.issubset(dna_chars):
                    dna_count += 1
                # Check if sequence contains protein-specific letters
                elif any(char in protein_chars for char in seq_set):
                    # Count how many are DNA vs Protein
                    dna_letters = len(seq_set.intersection(dna_chars))
                    protein_letters = len(seq_set.intersection(protein_chars))

                    # If more DNA letters, classify as DNA
                    if dna_letters > protein_letters:
                        dna_count += 1
                    else:
                        protein_count += 1
                else:
                    # Unknown characters, default to protein
                    protein_count += 1

        # Determine file type
        if dna_count > protein_count:
            fasta_type = "DNA"
        elif protein_count > dna_count:
            fasta_type = "Protein"
        else:
            # If tie, check the first sequence content
            fasta_type = "Unknown"

        if not seq_lens:
            avg_len = min_len = max_len = Q1 = Q2 = Q3 = N50 = N50_num = 0
        else:
            avg_len = sum(seq_lens)/num_seqs
            min_len = min(seq_lens)
            max_len = max(seq_lens)
            Q1 = statistics.quantiles(seq_lens, n=4)[0] if len(seq_lens) > 1 else seq_lens[0]
            Q2 = statistics.median(seq_lens)
            Q3 = statistics.quantiles(seq_lens, n=4)[2] if len(seq_lens) > 1 else seq_lens[0]

            sorted_lens = sorted(seq_lens, reverse=True)
            cumsum = 0
            half_sum = sum(seq_lens)/2
            N50 = N50_num = 0
            for i, l in enumerate(sorted_lens):
                cumsum += l
                if cumsum >= half_sum:
                    N50 = l
                    N50_num = i+1
                    break

        return {
            'fasta_seqkit_stat_info': {
                'format': 'FASTA',
                'type': fasta_type,
                'num_seqs': str(num_seqs),
                'sum_len': str(sum(seq_lens)),
                'min_len': str(min_len),
                'avg_len': str(int(avg_len)),
                'max_len': str(max_len),
                'Q1': str(int(Q1)),
                'Q2': str(int(Q2)),
                'Q3': str(int(Q3)),
                'sum_gap': str(sum_gap),
                'N50': str(N50),
                'N50_num': str(N50_num),
                'Q20(%)': '0',
                'Q30(%)': '0',
                'AvgQual': '0',
                'GC(%)': '0',
                'sum_n': '0'
            },
            'fasta_type': fasta_type,
            'fasta_num_seqs': num_seqs
        }

    def _get_uniprot(self, accession):
        """Make real API call to Uniprot"""
        try:
            url = f"https://rest.uniprot.org/uniprotkb/{accession}"
            response = requests.get(url, headers={"Accept": "application/json"})

            if response.status_code == 200:
                return response.json()
            else:
                return {"error": f"HTTP {response.status_code}"}
        except Exception as e:
            return {"error": str(e)}

    def _parse_uniprot_response(self, data):
        """Parse Uniprot API response"""
        if "error" in data:
            return data

        try:
            # Extract organism
            organism = data.get('organism', {}).get('scientificName', 'unknown')

            # Extract gene information
            gene_info = []
            for gene in data.get('genes', []):
                gene_entry = {}

                # Handle geneName
                if 'geneName' in gene:
                    gene_name = gene['geneName']
                    gene_name_dict = {'value': gene_name.get('value', 'unknown')}

                    # Add evidences if present
                    if 'evidences' in gene_name:
                        gene_name_dict['evidences'] = []
                        for evidence in gene_name['evidences']:
                            evidence_dict = {'evidenceCode': evidence.get('evidenceCode', 'unknown')}
                            if 'source' in evidence:
                                evidence_dict['source'] = evidence['source']
                            if 'id' in evidence:
                                evidence_dict['id'] = evidence['id']
                            gene_name_dict['evidences'].append(evidence_dict)

                    gene_entry['geneName'] = gene_name_dict

                # Handle synonyms
                if 'synonyms' in gene:
                    gene_entry['synonyms'] = []
                    for synonym in gene.get('synonyms', []):
                        gene_entry['synonyms'].append({'value': synonym.get('value', 'unknown')})

                if gene_entry:
                    gene_info.append(gene_entry)

            # If no gene info, create minimal structure
            if not gene_info:
                gene_info = [{'geneName': {'value': 'unknown'}}]

            # Extract sequence info
            sequence_data = data.get('sequence', {})
            sequence = sequence_data.get('value', '')

            sequence_info = {
                'value': sequence,
                'length': sequence_data.get('length', 0),
                'molWeight': sequence_data.get('molWeight', 0),
                'crc64': sequence_data.get('crc64', ''),
                'md5': sequence_data.get('md5', '')
            }

            # Determine type
            entry_type = data.get('entryType', 'unknown').lower()
            if 'reviewed' in entry_type:
                entry_type = 'protein'

            return {
                'organism': organism,
                'geneInfo': gene_info,
                'sequenceInfo': sequence_info,
                'type': entry_type
            }
        except Exception as e:
            return {"error": f"Parse error: {str(e)}"}

    def _get_ensembl(self, ensembl_id):
        """Make real API call to Ensembl"""
        try:
            # Clean ID (remove version if present)
            clean_id = ensembl_id.split('.')[0]
            url = f"https://rest.ensembl.org/lookup/id/{clean_id}"
            response = requests.get(url, headers={"Content-Type": "application/json"})

            if response.status_code == 200:
                return response.json()
            else:
                return {"error": f"HTTP {response.status_code}"}
        except Exception as e:
            return {"error": str(e)}

    def _parse_ensembl_response(self, data, sequence):
        """Parse Ensembl API response"""
        if "error" in data:
            return data

        try:
            # Extract gene information
            gene_info = [{
                'geneName': {
                    'value': data.get('display_name', data.get('id', 'unknown'))
                }
            }]

            # Add synonyms if available
            if 'display_name' in data and data['display_name'] != data.get('id'):
                gene_info[0]['synonyms'] = [{'value': data['id']}]

            # Calculate molecular weight for DNA (approximate)
            mol_weight = len(sequence) * 650 if sequence else 0

            sequence_info = {
                'value': sequence,
                'length': len(sequence),
                'molWeight': mol_weight,
                'crc64': '',
                'md5': hashlib.md5(sequence.encode()).hexdigest()
            }

            return {
                'organism': data.get('species', 'unknown'),
                'geneInfo': gene_info,
                'sequenceInfo': sequence_info,
                'type': 'dna'
            }
        except Exception as e:
            return {"error": f"Parse error: {str(e)}"}

    def _extract_uniprot_id(self, description):
        """Extract Uniprot ID from description - STRICT pattern for actual Uniprot IDs"""
        match = re.search(r'(?:sp|tr)\|([A-Z0-9]{6,10})\|', description)
        if match:
            return match.group(1)

        # Check for standalone Uniprot accessions that follow the exact pattern
        match = re.search(r'\b([OPQ][0-9][A-Z0-9]{3}[0-9])\b', description)
        if match:
            return match.group(1)

        match = re.search(r'\b([A-NR-Z][0-9][A-Z][A-Z0-9]{2}[0-9])\b', description)
        if match:
            return match.group(1)

        return None

    def _extract_ensembl_id(self, description):
        """Extract Ensembl ID from description"""
        patterns = [
            r'(ENS[GTP]\d{11}(?:\.\d+)?)',  # ENSG00000123456 format
            r'[|](ENS[GTP]\d{11}(?:\.\d+)?)[|]',  # |ENSG00000123456| format
        ]

        for pattern in patterns:
            match = re.search(pattern, description, re.I)
            if match:
                return match.group(1)
        return None

    def _get_database_info(self, id_string, db_type, sequence, description):
        """Get information from appropriate database"""
        if db_type == "uniprot":
            response = self._get_uniprot(id_string)
            return self._parse_uniprot_response(response)
        elif db_type == "ensembl":
            response = self._get_ensembl(id_string)
            return self._parse_ensembl_response(response, sequence)
        else:
            return {"error": "Unknown database type"}

    def biopython_parser(self, stats):
        """Parse FASTA and fetch data from databases - ONLY process valid IDs"""
        fasta_type = stats['fasta_type']

        # Determine which database to use based on file type
        if fasta_type == "Protein":
            db_name = "uniprot"
        elif fasta_type == "DNA":
            db_name = "ensembl"
        else:
            db_name = "unknown"

        output = {"DB_name": db_name}
        has_invalid_id = False

        # First pass: collect all valid sequences
        valid_sequences = []

        for record in SeqIO.parse(self.fasta_file, "fasta"):
            seq_val = str(record.seq)

            # Try to extract ID based on file type
            if db_name == "uniprot":
                id_value = self._extract_uniprot_id(record.description)
            elif db_name == "ensembl":
                id_value = self._extract_ensembl_id(record.description)
            else:
                id_value = None

            if id_value:
                # Store valid sequences
                valid_sequences.append((id_value, record.description, seq_val))
            else:
                # Track invalid IDs but DO NOT add to output
                has_invalid_id = True

        # Second pass: process only valid sequences
        for id_value, description, seq_val in valid_sequences:
            file_key = f"file_info_{id_value}"
            db_key = f"database_info_{id_value}"

            output[file_key] = {
                "description": description,
                "sequence": seq_val
            }

            # Make API call
            output[db_key] = self._get_database_info(id_value, db_name, seq_val, description)

        # Add warning only if there were sequences without valid IDs
        if has_invalid_id:
            self._warning.add("No ID match found.")

        return output

    def show_output(self, biopython_output):
        """Display formatted output matching test_data.txt format exactly"""
        # Print DB_name header
        print("> DB_name")
        if "DB_name" in biopython_output:
            print(f"\t{biopython_output['DB_name']}")

        # Process each key in the output
        for k, v in biopython_output.items():
            if k == "DB_name":
                continue

            print(k)

            if isinstance(v, dict):
                for subk, subv in v.items():
                    if isinstance(subv, dict):
                        print(f"\t{subk}")
                        for subk2, subv2 in subv.items():
                            if isinstance(subv2, dict):
                                print(f"\t\t{subk2}")
                                for subk3, subv3 in subv2.items():
                                    # Handle lists (like geneInfo)
                                    if isinstance(subv3, list):
                                        print(f"\t\t\t{subk3}")
                                        # Format the list as a JSON string
                                        list_str = json.dumps(subv3, default=str)
                                        print(f"\t\t\t\t{list_str}")
                                    else:
                                        print(f"\t\t\t{subk3}")
                                        print(f"\t\t\t\t{subv3}")
                            else:
                                print(f"\t\t{subk2}")
                                # Check if value is a list
                                if isinstance(subv2, list):
                                    list_str = json.dumps(subv2, default=str)
                                    print(f"\t\t\t{list_str}")
                                else:
                                    print(f"\t\t\t{subv2}")
                    else:
                        print(f"\t{subk}")
                        print(f"\t\t{subv}")
            else:
                print(f"\t{v}")

        # Print warnings if any
        if self._warning:
            print("WARNING")
            for warning in self._warning:
                print(f"\t{{{warning}}}")

In [27]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
print(stats)

{'fasta_seqkit_stat_info': {'format': 'FASTA', 'type': 'Protein', 'num_seqs': '2', 'sum_len': '456', 'min_len': '29', 'avg_len': '228', 'max_len': '427', 'Q1': '-70', 'Q2': '228', 'Q3': '526', 'sum_gap': '0', 'N50': '427', 'N50_num': '1', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0', 'GC(%)': '0', 'sum_n': '0'}, 'fasta_type': 'Protein', 'fasta_num_seqs': 2}


In [28]:
biopython = parser.biopython_parser(stats)

parser.show_output(biopython)

> DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'value': 'VDR', 'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}]}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCIT

In [30]:
parser_ensembl = MyFastaParser('ensembl_download_1.fasta')
stats_ensembl = parser_ensembl.seqkit_stats()
output_ensembl = parser_ensembl.biopython_parser(stats_ensembl)
parser_ensembl.show_output(output_ensembl)

> DB_name
	ensembl
file_info_ENST00000605284.1
	description
		ENST00000605284.1 cds chromosome:GRCh38:15:20011153:20011169:-1 gene:ENSG00000271336.1 gene_biotype:IG_D_gene transcript_biotype:IG_D_gene gene_symbol:IGHD1OR15-1A description:immunoglobulin heavy diversity 1/OR15-1A (non-functional) [Source:HGNC Symbol;Acc:HGNC:5487]
	sequence
		GGTATAACTGGAACAAC
database_info_ENST00000605284.1
	organism
		homo_sapiens
	geneInfo
		[{'geneName': {'value': 'IGHD1OR15-1A-201'}, 'synonyms': [{'value': 'ENST00000605284'}]}]
	sequenceInfo
		value
			GGTATAACTGGAACAAC
		length
			17
		molWeight
			11050
		crc64
			
		md5
			93e9e480cc759c7a75f44345eea0813a
	type
		dna
file_info_ENST00000604642.1
	description
		ENST00000604642.1 cds chromosome:GRCh38:15:20003840:20003862:-1 gene:ENSG00000270961.1 gene_biotype:IG_D_gene transcript_biotype:IG_D_gene gene_symbol:IGHD5OR15-5A description:immunoglobulin heavy diversity 5/OR15-5A (non-functional) [Source:HGNC Symbol;Acc:HGNC:5512]
	sequence
		GTGGATATAGT

In [31]:
parser_ensembl = MyFastaParser('ensembl_download_2.fasta')
stats_ensembl = parser_ensembl.seqkit_stats()
output_ensembl = parser_ensembl.biopython_parser(stats_ensembl)
parser_ensembl.show_output(output_ensembl)

> DB_name
	ensembl
file_info_ENST00000632684.1
	description
		ENST00000632684.1 cds chromosome:GRCh38:7:142786213:142786224:1 gene:ENSG00000282431.1 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:TRBD1 description:T cell receptor beta diversity 1 [Source:HGNC Symbol;Acc:HGNC:12158]
	sequence
		GGGACAGGGGGC
database_info_ENST00000632684.1
	organism
		homo_sapiens
	geneInfo
		[{'geneName': {'value': 'TRBD1-202'}, 'synonyms': [{'value': 'ENST00000632684'}]}]
	sequenceInfo
		value
			GGGACAGGGGGC
		length
			12
		molWeight
			7800
		crc64
			
		md5
			1f47b55923e2d23090f894c439974b55
	type
		dna
file_info_ENST00000710614.1
	description
		ENST00000710614.1 cds chromosome:GRCh38:7:142795705:142795720:1 gene:ENSG00000292271.1 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene description:T cell receptor beta diversity 2
	sequence
		GGGACTAGCGGGAGGG
database_info_ENST00000710614.1
	organism
		homo_sapiens
	geneInfo
		[{'geneName': {'value': 'ENST00000710614'}}]
	sequenceInfo


In [32]:
parser_uniprot = MyFastaParser('uniprot_download.fasta')
stats_uniprot = parser_uniprot.seqkit_stats()
output_uniprot = parser_uniprot.biopython_parser(stats_uniprot)
parser_uniprot.show_output(output_uniprot)

> DB_name
	uniprot
file_info_Q9R1A7
	description
		sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1
	sequence
		MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLKKLQLREEEYVLMQAISLFSPDRPGVVQRSVVDQLQERFALTLKAYIECSRPYPAHRFLFLKIMAVLTELRSINAQQTQQLLRIQDTHPFATPLMQELFSSTDG
database_info_Q9R1A7
	organism
		Rattus norvegicus
	geneInfo
		[{'geneName': {'value': 'Nr1i2'}, 'synonyms': [{'value': 'Pxr'}]}]
	sequenceInfo
		value
			MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIP